In [1]:
import os
import sys
import pandas as pd
import numpy as np
import yaml
import torch
from torch.utils.data import DataLoader

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(os.path.join(PROJECT_ROOT, "src"))
from deep_learning import (set_seed, get_device, TabularDataset, EmbeddingMLP,
                            prepare_tabular_arrays, train_mlp, predict_proba_mlp)
from modeling import tune_threshold, bootstrap_ci, evaluate_at_threshold
from ensemble import paired_bootstrap_delta_auc

with open(os.path.join(PROJECT_ROOT, "config", "config.yaml")) as f:
    config = yaml.safe_load(f)

SEED = config["seed"]
set_seed(SEED)
DEVICE = get_device()
print("Using device:", DEVICE)

PROCESSED_DIR = os.path.join(PROJECT_ROOT, config["paths"]["processed_dir"])
MODELS_ROOT = os.path.join(PROJECT_ROOT, config["paths"]["models_dir"])
TABLES_DIR = os.path.join(PROJECT_ROOT, config["paths"]["tables_dir"])
N_BOOTSTRAP = config["bootstrap"]["n_resamples"]

Using device: cpu


In [2]:
import joblib

def run_dl_model(dataset_tag, data_dir, numeric_cols, categorical_cols, label_col, models_dir):
    train = pd.read_csv(os.path.join(data_dir, "train.csv"))
    cal = pd.read_csv(os.path.join(data_dir, "calibration.csv"))
    test = pd.read_csv(os.path.join(data_dir, "test.csv"))

    (num_train, cat_train, num_cal, cat_cal, num_test, cat_test,
     embedding_cardinalities) = prepare_tabular_arrays(train, cal, test, numeric_cols, categorical_cols)

    y_train = train[label_col].values
    y_cal = cal[label_col].values
    y_test = test[label_col].values

    train_dataset = TabularDataset(num_train, cat_train, y_train)
    cal_dataset = TabularDataset(num_cal, cat_cal, y_cal)
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

    set_seed(SEED)
    model = EmbeddingMLP(n_numeric=len(numeric_cols), embedding_cardinalities=embedding_cardinalities,
                          hidden_dims=(64, 32), dropout=0.3)

    model, best_cal_auc = train_mlp(model, train_loader, cal_dataset, DEVICE, y_train, seed=SEED,
                                     max_epochs=100, patience=10, lr=1e-3, weight_decay=1e-4)
    print(f"[{dataset_tag}] Training done. Best calibration AUC during training: {best_cal_auc:.4f}")

    proba_cal = predict_proba_mlp(model, num_cal, cat_cal, DEVICE)
    thr, macro_f1 = tune_threshold(y_cal, proba_cal)
    print(f"[{dataset_tag}] Tuned threshold: {thr:.2f} (calibration macro-F1: {macro_f1:.4f})")

    proba_test = predict_proba_mlp(model, num_test, cat_test, DEVICE)
    metrics = evaluate_at_threshold(y_test, proba_test, thr)
    auc_point, auc_lo, auc_hi = bootstrap_ci(y_test, proba_test, n_resamples=N_BOOTSTRAP, seed=SEED)
    print(f"[{dataset_tag}] Test: Acc={metrics['accuracy']:.4f}  MacroF1={metrics['macro_f1']:.4f}  "
          f"AUC={metrics['auc_roc']:.4f} [{auc_lo:.4f}, {auc_hi:.4f}]")

    os.makedirs(models_dir, exist_ok=True)
    torch.save(model.state_dict(), os.path.join(models_dir, "mlp_model.pt"))

    summary = {
        "dataset": dataset_tag, "model": "embedding_mlp", "threshold": round(thr, 2),
        "accuracy": round(metrics["accuracy"], 4), "macro_f1": round(metrics["macro_f1"], 4),
        "auc_roc": round(metrics["auc_roc"], 4), "auc_ci_lower": round(auc_lo, 4),
        "auc_ci_upper": round(auc_hi, 4),
    }
    return summary, proba_test

In [3]:
numeric_cols_pk = ["Age", "Gestational Age", "Number of sons ", "Number of daughters", "Total Number of Children"]
categorical_cols_pk = ["Gravida", "Female Education", "Husband Education", "Working Status",
                        "Physical Health ", "Previous Miscarriage", "Sufficient Money for Basic Needs",
                        "Current Appereance Acceptance", "Family System", "Male Gender Preference",
                        "Relationship with Mother in-law"]

summary_pk_dl, proba_pk_dl = run_dl_model(
    "pakistan", os.path.join(PROCESSED_DIR, "pakistan"), numeric_cols_pk, categorical_cols_pk,
    "y_depression", os.path.join(MODELS_ROOT, "dl", "pakistan"))

[pakistan] Training done. Best calibration AUC during training: 0.5443
[pakistan] Tuned threshold: 0.47 (calibration macro-F1: 0.5297)
[pakistan] Test: Acc=0.5565  MacroF1=0.5259  AUC=0.5544 [0.5315, 0.5767]


In [4]:
numeric_cols_ar = ["WomenAge", "MariageAge", "NumberofPregnancy", "NumberofAbortions",
                    "CovidDiagL1", "CovidDiagL2", "SleepingAfter", "BMI_Pregnancy"]
categorical_cols_ar = ["part1_country", "Locality_first", "EducationLevel", "Work", "Health_Prob",
                        "StressduringCovid", "FamilyProblems", "FinancialProblem", "SocialProblem",
                        "PsychologicalProb", "WorkStress", "FamilyIncome1", "part1_current_preg_first"]

summary_ar_dep_dl, proba_ar_dep_dl = run_dl_model(
    "arab5_depression", os.path.join(PROCESSED_DIR, "arab5_depression"), numeric_cols_ar, categorical_cols_ar,
    "y_depression", os.path.join(MODELS_ROOT, "dl", "arab5_depression"))

[arab5_depression] Training done. Best calibration AUC during training: 0.6242
[arab5_depression] Tuned threshold: 0.48 (calibration macro-F1: 0.5939)
[arab5_depression] Test: Acc=0.6113  MacroF1=0.6113  AUC=0.6754 [0.6155, 0.7357]


In [5]:
summary_ar_anx_dl, proba_ar_anx_dl = run_dl_model(
    "arab5_anxiety", os.path.join(PROCESSED_DIR, "arab5_anxiety"), numeric_cols_ar, categorical_cols_ar,
    "y_anxiety", os.path.join(MODELS_ROOT, "dl", "arab5_anxiety"))

[arab5_anxiety] Training done. Best calibration AUC during training: 0.7446
[arab5_anxiety] Tuned threshold: 0.36 (calibration macro-F1: 0.7275)
[arab5_anxiety] Test: Acc=0.7958  MacroF1=0.7098  AUC=0.7733 [0.6861, 0.8519]


In [6]:
numeric_cols_ug = ["age_numeric", "otherChildren", "numStillBorn", "numMissCarry"]
categorical_cols_ug = ["homeDistrict", "occupation", "relationshipStatus", "spouseOccupation",
                        "polygamy", "education"]

train_ug = pd.read_csv(os.path.join(PROCESSED_DIR, "uganda", "train.csv"))
RARE_THRESHOLD = 30
district_counts = train_ug["homeDistrict"].value_counts()
keep_districts = set(district_counts[district_counts >= RARE_THRESHOLD].index)

for split_name in ["train.csv", "calibration.csv", "test.csv"]:
    df = pd.read_csv(os.path.join(PROCESSED_DIR, "uganda", split_name))
    df["homeDistrict"] = df["homeDistrict"].apply(lambda x: x if x in keep_districts else "Other")
    df.to_csv(os.path.join(PROCESSED_DIR, "uganda", split_name), index=False)

summary_ug_dl, proba_ug_dl = run_dl_model(
    "uganda", os.path.join(PROCESSED_DIR, "uganda"), numeric_cols_ug, categorical_cols_ug,
    "y_anxiety_binary", os.path.join(MODELS_ROOT, "dl", "uganda"))

[uganda] Training done. Best calibration AUC during training: 0.6757
[uganda] Tuned threshold: 0.51 (calibration macro-F1: 0.6294)
[uganda] Test: Acc=0.6364  MacroF1=0.6363  AUC=0.6906 [0.6708, 0.7102]


In [7]:
rf_results = pd.read_csv(os.path.join(TABLES_DIR, "proposed_model_final_results.csv"))
dl_results = pd.DataFrame([summary_pk_dl, summary_ar_dep_dl, summary_ar_anx_dl, summary_ug_dl])

comparison = pd.merge(
    rf_results[["dataset", "auc_roc", "auc_ci_lower", "auc_ci_upper"]].rename(
        columns={"auc_roc": "rf_auc", "auc_ci_lower": "rf_ci_lower", "auc_ci_upper": "rf_ci_upper"}),
    dl_results[["dataset", "auc_roc", "auc_ci_lower", "auc_ci_upper"]].rename(
        columns={"auc_roc": "dl_auc", "auc_ci_lower": "dl_ci_lower", "auc_ci_upper": "dl_ci_upper"}),
    on="dataset"
)
print(comparison)
comparison.to_csv(os.path.join(TABLES_DIR, "dl_vs_rf_comparison.csv"), index=False)

            dataset  rf_auc  rf_ci_lower  rf_ci_upper  dl_auc  dl_ci_lower  \
0          pakistan  0.5571       0.5353       0.5792  0.5544       0.5315   
1  arab5_depression  0.6867       0.6252       0.7468  0.6754       0.6155   
2     arab5_anxiety  0.7815       0.6958       0.8631  0.7733       0.6861   
3            uganda  0.6965       0.6770       0.7162  0.6906       0.6708   

   dl_ci_upper  
0       0.5767  
1       0.7357  
2       0.8519  
3       0.7102  


In [8]:
proba_pairs = {
    "pakistan": (proba_pk_dl, os.path.join(PROCESSED_DIR, "pakistan")),
    "arab5_depression": (proba_ar_dep_dl, os.path.join(PROCESSED_DIR, "arab5_depression")),
    "arab5_anxiety": (proba_ar_anx_dl, os.path.join(PROCESSED_DIR, "arab5_anxiety")),
    "uganda": (proba_ug_dl, os.path.join(PROCESSED_DIR, "uganda")),
}

delta_rows = []
for tag, (proba_dl, ddir) in proba_pairs.items():
    test_pred = pd.read_csv(os.path.join(ddir, "base_predictions_test.csv"))
    y_test = test_pred["y_true"].values
    proba_rf = test_pred["rf"].values
    delta, lo, hi = paired_bootstrap_delta_auc(y_test, proba_dl, proba_rf, n_resamples=N_BOOTSTRAP, seed=SEED)
    delta_rows.append({"dataset": tag, "delta_auc_dl_minus_rf": round(delta, 4),
                        "ci_lower": round(lo, 4), "ci_upper": round(hi, 4)})

delta_df = pd.DataFrame(delta_rows)
print(delta_df)
delta_df.to_csv(os.path.join(TABLES_DIR, "dl_vs_rf_delta_auc.csv"), index=False)

            dataset  delta_auc_dl_minus_rf  ci_lower  ci_upper
0          pakistan                -0.0029   -0.0186    0.0133
1  arab5_depression                -0.0109   -0.0395    0.0168
2     arab5_anxiety                -0.0080   -0.0417    0.0270
3            uganda                -0.0059   -0.0142    0.0020
